# 9.2 Bayesian networks and Markov networks: practice notebook

Draw uncertain knowledge as a graph. Directed Bayesian networks store small tables P(node | parents), d-separation reads independence from the arrows, and undirected Markov networks score worlds with factors and a partition function.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### A joint entry of the burglary network

In [ ]:
# The burglary network of AIMA figure 14.2. Each table gives P(variable = true | its parents).
parents = {"B": [], "E": [], "A": ["B", "E"], "J": ["A"], "M": ["A"]}
table = {"B": {(): 0.001}, "E": {(): 0.002},
         "A": {(True, True): 0.95, (True, False): 0.94, (False, True): 0.29, (False, False): 0.001},
         "J": {(True,): 0.90, (False,): 0.05}, "M": {(True,): 0.70, (False,): 0.01}}

def prob(var, value, world):
    key = tuple(world[p] for p in parents[var])
    p = table[var][key]
    return p if value else 1 - p

def joint(world):
    # chain rule shortened by the network: multiply P(x | parents(x)) for every variable
    result = 1
    for var in ["B", "E", "A", "J", "M"]:
        result = result * prob(var, world[var], world)
    return result

w = {"B": False, "E": False, "A": True, "J": True, "M": True}
print("P(j, m, a, not b, not e) =", round(joint(w), 6))

count = 0
for var in parents:
    count = count + 2 ** len(parents[var])
print("numbers in the network:", count, " numbers in the full joint:", 2 ** 5 - 1)

### d-separation by the moral ancestral graph

In [ ]:
# d-separation by the moral ancestral graph (Lauritzen and others, 1990):
# X and Y are d-separated by Z exactly when they are not connected in the graph built below.
parents = {"Cloudy": [], "Sprinkler": ["Cloudy"], "Rain": ["Cloudy"], "WetGrass": ["Sprinkler", "Rain"]}

def d_separated(x, y, z):
    # 1. keep only x, y, z and their ancestors
    keep = [x, y] + z
    i = 0
    while i < len(keep):
        for p in parents[keep[i]]:
            if p not in keep:
                keep.append(p)
        i = i + 1
    # 2. moralise: join each child to its parents, and the parents of a child to each other
    edges = []
    for v in keep:
        ps = parents[v]
        for p in ps:
            edges.append((p, v))
        for a in ps:
            for b in ps:
                if a < b:
                    edges.append((a, b))
    # 3. delete the evidence z, then look for a path from x to y
    seen = [x]
    todo = [x]
    while todo:
        u = todo.pop()
        for a, b in edges:
            for s, t in [(a, b), (b, a)]:
                if s == u and t not in z and t not in seen:
                    seen.append(t)
                    todo.append(t)
    return y not in seen

print("Sprinkler, Rain | Cloudy:", d_separated("Sprinkler", "Rain", ["Cloudy"]))
print("Sprinkler, Rain | Cloudy, WetGrass:", d_separated("Sprinkler", "Rain", ["Cloudy", "WetGrass"]))
print("Cloudy, WetGrass | Sprinkler, Rain:", d_separated("Cloudy", "WetGrass", ["Sprinkler", "Rain"]))

### The partition function of the misconception network

In [ ]:
# The misconception Markov network of Koller and Friedman (2009): A - B - C - D - A.
# phi[(X, Y)][(x, y)] scores how well x and y fit together.
phi = {("A", "B"): {(0, 0): 30, (0, 1): 5, (1, 0): 1, (1, 1): 10},
       ("B", "C"): {(0, 0): 100, (0, 1): 1, (1, 0): 1, (1, 1): 100},
       ("C", "D"): {(0, 0): 1, (0, 1): 100, (1, 0): 100, (1, 1): 1},
       ("D", "A"): {(0, 0): 100, (0, 1): 1, (1, 0): 1, (1, 1): 100}}

Z = 0
best, best_world = 0, None
for a in [0, 1]:
    for b in [0, 1]:
        for c in [0, 1]:
            for d in [0, 1]:
                w = {"A": a, "B": b, "C": c, "D": d}
                score = 1
                for (x, y), f in phi.items():
                    score = score * f[(w[x], w[y])]
                Z = Z + score
                if score > best:
                    best, best_world = score, (a, b, c, d)
print("Z =", Z)
print("most likely world (A, B, C, D) =", best_world, " P =", round(best / Z, 3))

## Your turn

### Task 1: The probability of one world

Write `joint(world)` for the burglary network. world maps "B", "E", "A", "J", "M" to True or False. Multiply P(x | parents) over all five variables.

Hint: For each variable, build the key from the parents’ values, look up P(true), and use 1 − p when the variable is false.

In [ ]:
PARENTS = {"B": [], "E": [], "A": ["B", "E"], "J": ["A"], "M": ["A"]}
TABLE = {"B": {(): 0.001}, "E": {(): 0.002},
         "A": {(True, True): 0.95, (True, False): 0.94, (False, True): 0.29, (False, False): 0.001},
         "J": {(True,): 0.90, (False,): 0.05}, "M": {(True,): 0.70, (False,): 0.01}}

def joint(world):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("round(joint({'B': False, 'E': False, 'A': True, 'J': True, 'M': True}), 6)", 0.000628), ("round(joint({'B': False, 'E': False, 'A': False, 'J': False, 'M': False}), 4)", 0.9367), ("round(joint({'B': True, 'E': True, 'A': True, 'J': True, 'M': True}) * 10 ** 6, 3)", 1.197)])

### Task 2: Count the numbers a network needs

Write `count_params(parents)` for a network of true or false nodes: the sum over nodes of 2 to the power of the number of parents.

Hint: Loop over the nodes and add 2 ** len(parents[node]).

In [ ]:
def count_params(parents):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("count_params({'B': [], 'E': [], 'A': ['B', 'E'], 'J': ['A'], 'M': ['A']})", 10), ("count_params({'C': [], 'S': ['C'], 'R': ['C'], 'W': ['S', 'R']})", 9), ("count_params({'X': []})", 1), ("count_params({'A': [], 'B': [], 'C': [], 'D': ['A', 'B', 'C']})", 11)])

### Task 3: The Markov blanket

Write `blanket(parents, x)` that returns the sorted list of nodes in the Markov blanket of x: its parents, its children, and the other parents of its children.

Hint: A child is any node that lists x among its parents.

In [ ]:
def blanket(parents, x):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("blanket({'B': [], 'E': [], 'A': ['B', 'E'], 'J': ['A'], 'M': ['A']}, 'A')", ['B', 'E', 'J', 'M']), ("blanket({'B': [], 'E': [], 'A': ['B', 'E'], 'J': ['A'], 'M': ['A']}, 'B')", ['A', 'E']), ("blanket({'C': [], 'S': ['C'], 'R': ['C'], 'W': ['S', 'R']}, 'S')", ['C', 'R', 'W']), ("blanket({'C': [], 'S': ['C'], 'R': ['C'], 'W': ['S', 'R']}, 'W')", ['R', 'S'])])

### Task 4: d-separation with the moral ancestral graph

Write `d_separated(parents, x, y, z)`: keep x, y, the list z and all their ancestors, moralise (join each node to its parents and parents of a common child to each other), delete z, and return True when no path joins x and y.

Hint: Grow the list of ancestors first. Then build the moral edges, and search from x without stepping on any node in z.

In [ ]:
def d_separated(parents, x, y, z):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("d_separated({'B': [], 'E': [], 'A': ['B', 'E'], 'J': ['A'], 'M': ['A']}, 'B', 'E', [])", True), ("d_separated({'B': [], 'E': [], 'A': ['B', 'E'], 'J': ['A'], 'M': ['A']}, 'B', 'E', ['J'])", False), ("d_separated({'B': [], 'E': [], 'A': ['B', 'E'], 'J': ['A'], 'M': ['A']}, 'J', 'M', ['A'])", True), ("d_separated({'C': [], 'S': ['C'], 'R': ['C'], 'W': ['S', 'R']}, 'S', 'R', ['C', 'W'])", False)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: The probability of one world

Test 2: nothing happens, 0.999 × 0.998 × 0.999 × 0.95 × 0.99 ≈ 0.9367.

In [ ]:
PARENTS = {"B": [], "E": [], "A": ["B", "E"], "J": ["A"], "M": ["A"]}
TABLE = {"B": {(): 0.001}, "E": {(): 0.002},
         "A": {(True, True): 0.95, (True, False): 0.94, (False, True): 0.29, (False, False): 0.001},
         "J": {(True,): 0.90, (False,): 0.05}, "M": {(True,): 0.70, (False,): 0.01}}

def joint(world):
    result = 1
    for var in PARENTS:
        key = tuple(world[p] for p in PARENTS[var])
        p = TABLE[var][key]
        if world[var]:
            result = result * p
        else:
            result = result * (1 - p)
    return result

In [ ]:
check([("round(joint({'B': False, 'E': False, 'A': True, 'J': True, 'M': True}), 6)", 0.000628), ("round(joint({'B': False, 'E': False, 'A': False, 'J': False, 'M': False}), 4)", 0.9367), ("round(joint({'B': True, 'E': True, 'A': True, 'J': True, 'M': True}) * 10 ** 6, 3)", 1.197)])

### Solution to task 2: Count the numbers a network needs

Test 2 is the sprinkler network: 1 + 2 + 2 + 4 = 9.

In [ ]:
def count_params(parents):
    total = 0
    for node in parents:
        total = total + 2 ** len(parents[node])
    return total

In [ ]:
check([("count_params({'B': [], 'E': [], 'A': ['B', 'E'], 'J': ['A'], 'M': ['A']})", 10), ("count_params({'C': [], 'S': ['C'], 'R': ['C'], 'W': ['S', 'R']})", 9), ("count_params({'X': []})", 1), ("count_params({'A': [], 'B': [], 'C': [], 'D': ['A', 'B', 'C']})", 11)])

### Solution to task 3: The Markov blanket

Test 2: Earthquake is in the blanket of Burglary because both are parents of Alarm.

In [ ]:
def blanket(parents, x):
    out = []
    for p in parents[x]:
        out.append(p)
    for child in parents:
        if x in parents[child]:
            if child not in out:
                out.append(child)
            for p in parents[child]:
                if p != x and p not in out:
                    out.append(p)
    return sorted(out)

In [ ]:
check([("blanket({'B': [], 'E': [], 'A': ['B', 'E'], 'J': ['A'], 'M': ['A']}, 'A')", ['B', 'E', 'J', 'M']), ("blanket({'B': [], 'E': [], 'A': ['B', 'E'], 'J': ['A'], 'M': ['A']}, 'B')", ['A', 'E']), ("blanket({'C': [], 'S': ['C'], 'R': ['C'], 'W': ['S', 'R']}, 'S')", ['C', 'R', 'W']), ("blanket({'C': [], 'S': ['C'], 'R': ['C'], 'W': ['S', 'R']}, 'W')", ['R', 'S'])])

### Solution to task 4: d-separation with the moral ancestral graph

Test 2: J is a descendant of the collider A, so A and J are kept, B and E get married, and B reaches E.

In [ ]:
def d_separated(parents, x, y, z):
    keep = [x, y] + z
    i = 0
    while i < len(keep):
        for p in parents[keep[i]]:
            if p not in keep:
                keep.append(p)
        i = i + 1
    edges = []
    for v in keep:
        for p in parents[v]:
            edges.append((p, v))
        for a in parents[v]:
            for b in parents[v]:
                if a < b:
                    edges.append((a, b))
    seen = [x]
    todo = [x]
    while todo:
        u = todo.pop()
        for a, b in edges:
            for s, t in [(a, b), (b, a)]:
                if s == u and t not in z and t not in seen:
                    seen.append(t)
                    todo.append(t)
    return y not in seen

In [ ]:
check([("d_separated({'B': [], 'E': [], 'A': ['B', 'E'], 'J': ['A'], 'M': ['A']}, 'B', 'E', [])", True), ("d_separated({'B': [], 'E': [], 'A': ['B', 'E'], 'J': ['A'], 'M': ['A']}, 'B', 'E', ['J'])", False), ("d_separated({'B': [], 'E': [], 'A': ['B', 'E'], 'J': ['A'], 'M': ['A']}, 'J', 'M', ['A'])", True), ("d_separated({'C': [], 'S': ['C'], 'R': ['C'], 'W': ['S', 'R']}, 'S', 'R', ['C', 'W'])", False)])